In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
data = pd.read_csv('../datasets/cleaned.csv')
print(f"Dataset shape: {data.shape}")

target_col = 'Overall'

numeric_cols = data.select_dtypes(include=['int64', 'float64']).columns.tolist()
feature_cols = [col for col in numeric_cols if col != target_col]

data = data.dropna(subset=[target_col])
X = data[feature_cols]
y = data[target_col]

X = X.fillna(X.median())

print(f"Number of numeric features: {len(feature_cols)}")

Dataset shape: (18207, 56)
Number of numeric features: 46


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline(steps=[
    ('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor())
])

param_grid = {
    'knn__n_neighbors': [3, 5, 7, 9, 11],
    'knn__weights': ['uniform', 'distance'],
    'knn__p': [1, 2]
}

knn_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=1
)

knn_search.fit(X_train, y_train)

print('Best params:', knn_search.best_params_)
print('Best CV RMSE:', (-knn_search.best_score_) ** 0.5)

Best params: {'knn__n_neighbors': 11, 'knn__p': 1, 'knn__weights': 'distance'}
Best CV RMSE: 1.6284445508559864


In [9]:
best_model = knn_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"Test MAE: {mae:.3f}")
print(f"Test RMSE: {rmse:.3f}")
print(f"Test R^2: {r2:.3f}")

Test MAE: 1.241
Test RMSE: 1.588
Test R^2: 0.946


In [11]:
outer_cv = KFold(n_splits=5, shuffle=True, random_state=42)
nested_mse_scores = []

for train_idx, val_idx in outer_cv.split(X, y):
    X_train_cv, X_val_cv = X.iloc[train_idx], X.iloc[val_idx]
    y_train_cv, y_val_cv = y.iloc[train_idx], y.iloc[val_idx]

    inner_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        scoring='neg_mean_squared_error',
        cv=5,
        n_jobs=-1
    )
    inner_search.fit(X_train_cv, y_train_cv)

    preds_outer = inner_search.predict(X_val_cv)
    mse_outer = mean_squared_error(y_val_cv, preds_outer)
    nested_mse_scores.append(mse_outer)

print(f"Nested CV MSE per fold: {nested_mse_scores}")
print(f"Nested CV Mean MSE: {np.mean(nested_mse_scores)}")
print(f"Nested CV RMSE: {np.sqrt(np.mean(nested_mse_scores))}")

Nested CV MSE per fold: [2.5190814047346426, 2.58830053869779, 2.5733927146611624, 2.7032762222869007, 2.712889402926573]
Nested CV Mean MSE: 2.6193880566614136
Nested CV RMSE: 1.6184523646562519
